# Shot-model validation across seasons

This experiment trains on **2023–2024**, validates on **2025**, and reserves
**2026** for final testing. It retains the original current-state logistic regression:
the model estimates whether a subsequent shot will occur within the same
possession. It does not require a goal.

Run cells in order after following the multi-season preparation commands in the
README and installing `.[notebook,model]`. Saved outputs provide a reference.
The earlier notebooks and their outputs document the initial 2023 experiment.

The [baseline specification](../docs/logistic-baseline.md) and
[coverage audit](../docs/season-coverage.md) describe the assumptions and limitations.

## 1. Environment and modelling functions

The setup locates the repository from the working directory and imports the
loading and modelling functions shared with the command-line workflow. `Random`
generates reproducible baseline scores independent of the labels.

In [1]:
import json
from pathlib import Path
from random import Random

import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, average_precision_score

from afl_opposition.baseline import load_preprocessed_seasons, train_baseline, save_baseline
from afl_opposition.examples import season_split
from afl_opposition.preprocess import FEATURE_COLUMNS

project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data/sources.json").is_file() and (p / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
print("Model inputs:", FEATURE_COLUMNS)
from afl_opposition.evaluation import confusion_report

Model inputs: ['action_type', 'x_norm', 'y_norm', 'gap_seconds', 'elapsed_seconds']


## 2. Pinned season coverage

The coverage audit compares the source archives with the concluded-match fixture catalogue.
The 2026 event archive exists. One 2024 match is missing, and a small number of
quarters are excluded for reversed timestamps. Match presence alone does not
guarantee complete event annotations.

This cell reads audit metadata; it does not load test examples or calculate
test performance. The coverage record identifies the exact source checksums.

In [2]:
coverage_record = json.loads((project_root / "data/season-coverage.json").read_text())
coverage = pd.DataFrame(coverage_record["seasons"])
coverage["role"] = coverage.season.map({2023: "train", 2024: "train", 2025: "validation", 2026: "test reserved"})
coverage["quarantined_quarter_count"] = coverage.quarantined_quarters.map(len)
display(coverage[["season", "role", "raw_rows", "event_matches", "fixture_matches",
                  "quarantined_quarter_count", "quarantined_rows"]])
print("Coverage checked on:", coverage_record["checked_on"])
for missing in coverage_record["seasons"]:
    for match in missing["missing_matches"]:
        print("Missing match:", missing["season"], match["match_id"],
              match["home_team_name"], "vs", match["away_team_name"])

,season,role,raw_rows,event_matches,fixture_matches,quarantined_quarter_count,quarantined_rows
0,2023,train,442482,216,216,0,0
1,2024,train,426754,215,216,1,589
2,2025,validation,430827,216,216,2,1027
3,2026,test reserved,447945,218,218,3,1616


Coverage checked on: 2026-10-11
Missing match: 2024 CD_M20240142309 Western Bulldogs vs North Melbourne


## 3. Development data and label audits

The development dataset combines 2023, 2024 and 2025. Each source checksum
and preprocessing record is verified before the tables are combined. Quarter exclusions retain all source
annotations in the audit files.

The loader checks auxiliary `Shot At Goal` markers against preceding flagged
kicks. A conflict excludes every possession touching that raw chain rather
than inventing a missing shot label. The output reports these exclusions.
Unknown action descriptions continue to trigger the existing review rules.

In [3]:
DEVELOPMENT_SEASONS = (2023, 2024, 2025)
events, possessions, provenance = load_preprocessed_seasons(project_root, DEVELOPMENT_SEASONS)
audit_rows = []
for season, source in provenance["sources"].items():
    audit_rows.append({
        "season": int(season),
        "quarantined_quarters": len(source["quarantined_quarters"]),
        "quarantined_rows": source["quarantined_rows"],
        "verified_shot_markers": source["verified_retrospective_shot_markers"],
        "conflicting_shot_markers": source["conflicting_shot_markers"],
        "shot_conflict_possessions": source["shot_conflict_possessions"],
    })
display(pd.DataFrame(audit_rows))
assert set(events.season) == set(DEVELOPMENT_SEASONS)
print("2026 examples loaded:", False)

,season,quarantined_quarters,quarantined_rows,verified_shot_markers,conflicting_shot_markers,shot_conflict_possessions
0,2023,0,0,2000,0,0
1,2024,1,589,1760,1,1
2,2025,2,1027,1843,1,2


2026 examples loaded: False


## 4. Season-level partitions

The split assigns 2023–2024 to training and 2025 to validation. Every match, possession and
prefix stays in one partition. Assignments reject overlapping seasons,
unassigned seasons and dates that contradict their season.

The 2026 reservation is part of the split contract, but test examples are not
needed for development. Validation can guide future model choices; test
performance should be inspected only after those choices are frozen.

In [4]:
TRAIN_SEASONS = (2023, 2024)
VALIDATION_SEASON = 2025
TEST_SEASON = 2026
matches = season_split(events, train_seasons=TRAIN_SEASONS,
                       validation_season=VALIDATION_SEASON, test_season=TEST_SEASON)
display(matches.groupby(["season", "split"]).agg(
    matches=("match_id", "size"), first_date=("date", "min"), last_date=("date", "max"),
).reset_index())
assert matches.match_id.is_unique
assert matches.loc[matches.split.eq("train"), "date"].max() < matches.loc[matches.split.eq("validation"), "date"].min()
print("Reserved test season:", TEST_SEASON)
DECISION_THRESHOLD = 0.5  # Reference threshold; not selected through optimisation.

,season,split,matches,first_date,last_date
0,2023,train,216,2023-03-16,2023-09-30
1,2024,train,215,2024-03-07,2024-09-28
2,2025,validation,216,2025-03-07,2025-09-27


Reserved test season: 2026


## 5. Logistic regression fitting

The label constructor produces pre-shot examples, excluding reviewed or censored
episodes. Positive states precede the first explicit shot; the shot itself is
excluded. The one-hot encoder, scaler and classifier fit training examples only.
The classifier still uses L2 regularisation with `C=1`; no tuning occurs here.

Each eligible possession has total sample weight 1, divided across its prefixes.
This prevents long possessions from receiving more total weight simply because
they contain more actions. The table reports both example and possession counts.

In [5]:
result = train_baseline(events, possessions, train_seasons=TRAIN_SEASONS,
                        validation_season=VALIDATION_SEASON, test_season=TEST_SEASON,
                        decision_threshold=DECISION_THRESHOLD)
display(result.examples.groupby(["season", "split"]).agg(
    action_states=("event_id", "size"), possessions=("possession_id", "nunique"),
).reset_index())
display(result.possession_audit.groupby(["split", "status"]).size().rename("possessions").reset_index())
print("Encoded features:", result.summary["encoded_feature_count"])
print("Constant probability learned from training:", round(result.summary["constant_probability"], 4))
assert "test" not in set(result.predictions.split)

,season,split,action_states,possessions
0,2023,train,285771,52429
1,2024,train,277528,50649
2,2025,validation,281476,50744


,split,status,possessions
0,train,eligible,103078
1,train,no_pre_shot_action,60
2,train,preprocessing_review,7067
3,train,score_without_shot_flag,3
4,train,shot_marker_conflict,1
5,validation,eligible,50744
6,validation,no_pre_shot_action,42
7,validation,preprocessing_review,3245
8,validation,score_without_shot_flag,2
9,validation,shot_marker_conflict,2


Encoded features: 16
Constant probability learned from training: 0.1975


## 5a. Training objective

The fitted objective is **possession-weighted binary cross-entropy plus L2
regularisation**. Here $p_i=\sigma(\beta^T x_i+b)$ is the predicted probability,
$s_i$ is the sample weight and $S=\sum_i s_i$. With the configured solver, the objective is

$$
J(\beta,b)=-\frac{1}{S}\sum_i s_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
+\frac{\lVert\beta\rVert_2^2}{2CS}.
$$

For a possession with $n_j$ eligible prefixes, each prefix receives $s_i=1/n_j$.
Each possession contributes total weight 1; this does not balance the classes.
`C=1` controls inverse regularisation strength. The intercept is unpenalised.
`lbfgs` is the optimiser. See the [scikit-learn objective](https://scikit-learn.org/stable/modules/linear_model.html#binary-case).

AP, F1 and confusion counts are evaluation quantities. The model does not
optimise them directly. The reported validation log loss excludes the L2 term.
The cell separates the training data loss from its regularisation contribution.
The decision threshold does not enter either term.

In [6]:
train_rows = result.examples.loc[result.examples.split.eq("train")]
weight_sum = float(train_rows.sample_weight.sum())
training_bce = float(result.metrics.loc[
    result.metrics.split.eq("train") & result.metrics.model.eq("logistic"), "log_loss"
].iloc[0])
l2_term = float(result.coefficients.coefficient.pow(2).sum()) / (2 * result.summary["C"] * weight_sum)
display(pd.DataFrame({
    "component": ["Weighted training BCE", "L2 contribution", "Total training objective"],
    "value": [training_bce, l2_term, training_bce + l2_term],
}).round(8))
print("Sum of training weights:", round(weight_sum))
print("Eligible training possessions:", train_rows.possession_id.nunique())

,component,value
0,Weighted training BCE,0.399834
1,L2 contribution,0.000033
2,Total training objective,0.399868


Sum of training weights: 103078
Eligible training possessions: 103078


## 6. Validation probability metrics

The reference model assigns a constant probability equal to the weighted training
shot rate. Higher average precision is better; lower log loss and Brier score
are better. All metrics use the same possession weights.

Average precision describes ranking across thresholds. It is not accuracy or
precision at one decision threshold. For a constant score, validation AP equals
the validation positive-class prevalence, even if the constant probability was
learned from a different training prevalence.

The earlier AP of about 0.528 came from later **2023** matches. This table uses
**2025** validation. Different evaluation populations mean the difference is not
a controlled measurement of the benefit of adding training data.

In [7]:
validation_metrics = result.metrics.loc[result.metrics.split.eq("validation")]
display(validation_metrics[["model", "observed_shot_rate", "average_precision", "log_loss", "brier_score"]].round(4))
validation = result.predictions.loc[result.predictions.split.eq("validation")].copy()
y = validation.target
weights = validation.sample_weight
prevalence = float((y * weights).sum() / weights.sum())
print(f"Weighted validation shot prevalence: {prevalence:.2%}")
print("Test performance: not evaluated")

,model,observed_shot_rate,average_precision,log_loss,brier_score
2,constant,0.2042,0.2042,0.5063,0.1625
3,logistic,0.2042,0.5387,0.4112,0.1295


Weighted validation shot prevalence: 20.42%
Test performance: not evaluated


## 6a. Validation confusion matrices

A state is classified as shot-positive when its probability is **greater than
or equal to `DECISION_THRESHOLD`**, initially 0.5. Rows show actual outcomes; columns show
predictions. Thus the cells are `[[TN, FP], [FN, TP]]`. These are predictions
of a subsequent shot at an action state, not predictions made on the shot itself.

The first matrix contains raw action-state counts. The second sums the same
possession weights used during fitting and probability evaluation. Weighted
cells can be fractional and sum to the number of eligible possessions, but
they are not counts of uniquely classified possessions: prefixes from one
possession can enter different prediction columns.

Recall and specificity describe class-level performance alongside accuracy
and precision. At the default threshold the model misses most positive states. Balanced accuracy
averages recall and specificity; F1 combines positive-class precision and recall.
Different raw and weighted results arise from their different units of weight.

The reference threshold is not an established operating point. Change it in
section 4 and rerun the following cells to inspect another threshold. Such a
change affects confusion counts, not the probability ranking or training loss.
Any eventual operational threshold requires validation-based selection,
with the test season reserved.

In [8]:
logistic_confusion = result.confusion.loc[
    result.confusion.split.eq("validation") & result.confusion.model.eq("logistic")
]
for record in logistic_confusion.itertuples(index=False):
    matrix = pd.DataFrame(
        [[record.tn, record.fp], [record.fn, record.tp]],
        index=["Actual no shot", "Actual shot"],
        columns=["Predict no shot", "Predict shot"],
    )
    print(record.weighting, "| threshold:", record.threshold)
    display(matrix.astype(int) if record.weighting == "unweighted" else matrix.round(2))

rate_columns = ["accuracy", "precision", "recall", "specificity", "f1", "balanced_accuracy"]
print("Rates below are percentages:")
display((logistic_confusion.set_index("weighting")[rate_columns].T * 100).round(2))

unweighted | threshold: 0.5


,Predict no shot,Predict shot
Actual no shot,195412,9935
Actual shot,59808,16321


possession_weighted | threshold: 0.5


,Predict no shot,Predict shot
Actual no shot,38860.65,1521.35
Actual shot,7431.15,2930.85


Rates below are percentages:


weighting,unweighted,possession_weighted
accuracy,75.22,82.36
precision,62.16,65.83
recall,21.44,28.28
specificity,95.16,96.23
f1,31.88,39.57
balanced_accuracy,58.30,62.26


## 7. Expected confusion matrix for a random classifier

This illustrative population contains 100 labelled states: 20 shot-positive
and 80 shot-negative. Independent fair-coin predictions classify half of each
group as positive **in expectation**. The table reports expected counts;
individual random trials vary around them.

Accuracy is the fraction of all predictions that are correct. Precision is
the positive fraction among predicted positives; recall is the fraction of
actual positives detected. The random classifier has expected accuracy and
recall of 0.5, while its population precision equals positive prevalence.

In [9]:
tp, fp, fn, tn = 10, 40, 10, 40
display(pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index=["Predict shot", "Predict no shot"], columns=["Actual shot", "Actual no shot"],
))
print(f"Accuracy = (TP + TN) / all = {(tp + tn) / (tp + fp + fn + tn):.0%}")
print(f"Precision = TP / (TP + FP) = {tp / (tp + fp):.0%}")
print(f"Recall = TP / (TP + FN) = {tp / (tp + fn):.0%}")

,Actual shot,Actual no shot
Predict shot,10,40
Predict no shot,10,40


Accuracy = (TP + TN) / all = 50%
Precision = TP / (TP + FP) = 20%
Recall = TP / (TP + FN) = 50%


## 8. Random-score baseline on validation data

The random baseline generates independent uniform scores with a fixed seed.
Thresholding at 0.5 produces fair-coin class predictions for accuracy, precision
and recall. Average precision uses the original continuous scores.

The random row should have accuracy and recall near 0.5, but precision and AP
near the weighted shot prevalence. Small deviations are sampling noise. The
constant score ranks everything equally, giving AP exactly equal to prevalence.

`DECISION_THRESHOLD` defaults to 0.5 and is a reference rather than a selected
operational threshold. At 0.5 the random hard predictions are fair coin flips. The constant training probability is below 0.5, so it predicts no
shots at that threshold; its precision is undefined and displayed as missing.
High accuracy from predicting the majority class does not imply useful shot
detection. AP averages precision using increases in recall across thresholds;
it is not the arithmetic mean of these displayed precision values.

In [10]:
rng = Random(42)
random_scores = pd.Series([rng.random() for _ in range(len(validation))], index=validation.index)
comparisons = {
    "Independent random scores": random_scores,
    "Constant training prevalence": validation.constant_probability,
    "Logistic regression": validation.shot_probability,
}
metric_rows = []
for name, scores in comparisons.items():
    predicted = scores.ge(DECISION_THRESHOLD)
    metric_rows.append({
        "method": name,
        "accuracy_at_threshold": accuracy_score(y, predicted, sample_weight=weights),
        "precision_at_threshold": precision_score(y, predicted, sample_weight=weights, zero_division=float("nan")),
        "recall_at_threshold": recall_score(y, predicted, sample_weight=weights),
        "average_precision": average_precision_score(y, scores, sample_weight=weights),
    })
display(pd.DataFrame(metric_rows).round(4))
assert abs(average_precision_score(y, [0.5] * len(y), sample_weight=weights) - prevalence) < 1e-10
print(f"Giving every example score 0.5 also yields AP = {prevalence:.4f}.")
print("Decision threshold:", DECISION_THRESHOLD)

,method,accuracy_at_threshold,precision_at_threshold,recall_at_threshold,average_precision
0,Independent random scores,0.4987,0.2040,0.5013,0.2033
1,Constant training prevalence,0.7958,NaN,0.0000,0.2042
2,Logistic regression,0.8236,0.6583,0.2828,0.5387


Giving every example score 0.5 also yields AP = 0.2042.
Decision threshold: 0.5


## 8a. Baseline confusion-matrix comparison

The constant predictor, independent random scores and logistic regression
share the same validation rows, possession weights and decision threshold.
Each pair of rows below forms a two-by-two matrix, with actual outcomes on
rows and predictions on columns. Random-score results are one reproducible
simulation, not exact expected counts.

At 0.5 the constant predictor always predicts no shot, giving zero recall and
undefined positive precision. The coin flips identify about half of the
positive weight but also flag about half of the negative weight. Logistic
regression produces fewer false positives while missing more positives at
this threshold. The choice depends on how the predictions will be used.

In [11]:
comparison_predictions = validation.assign(random_probability=random_scores)
comparison_confusion = confusion_report(
    comparison_predictions, threshold=DECISION_THRESHOLD,
    probability_columns={"constant": "constant_probability", "random": "random_probability", "logistic": "shot_probability"},
)
weighted_comparison = comparison_confusion.loc[comparison_confusion.weighting.eq("possession_weighted")]
matrix_rows = []
for record in weighted_comparison.itertuples(index=False):
    matrix_rows.extend([
        {"model": record.model, "actual": "No shot", "Predict no shot": record.tn, "Predict shot": record.fp},
        {"model": record.model, "actual": "Shot", "Predict no shot": record.fn, "Predict shot": record.tp},
    ])
display(pd.DataFrame(matrix_rows).set_index(["model", "actual"]).round(2))
display(weighted_comparison[["model", "threshold", *rate_columns]].round(4))

Predict no shot  Predict shot
model    actual                                
constant No shot         40382.00          0.00
         Shot            10362.00          0.00
random   No shot         20110.19      20271.81
         Shot             5167.74       5194.26
logistic No shot         38860.65       1521.35
         Shot             7431.15       2930.85

,model,threshold,accuracy,precision,recall,specificity,f1,balanced_accuracy
1,constant,0.5,0.7958,NaN,0.0000,1.0000,0.0000,0.5000
3,random,0.5,0.4987,0.2040,0.5013,0.4980,0.2900,0.4996
5,logistic,0.5,0.8236,0.6583,0.2828,0.9623,0.3957,0.6226


## 9. Development artifacts and limitations

The export contains the fitted pipeline, example and prediction tables, metrics,
calibration, coverage, raw and weighted confusion matrices, and source provenance.
These generated artifacts are excluded from Git. This cell explicitly overwrites this experiment's previous generated
outputs; raw data and earlier notebooks are preserved.

At this development stage, 2025 supports model selection and 2026 remains reserved.
Results apply to eligible archived possessions and depend on annotation quality
and the inferred possession rules. They do not establish live-match reliability.

In [12]:
output_dir = project_root / "artifacts/logistic-regression-2023-2025"
save_baseline(result, output_dir, provenance, overwrite=True)
print("Saved:", output_dir.relative_to(project_root).as_posix())
print("Training seasons:", list(TRAIN_SEASONS))
print("Validation season:", VALIDATION_SEASON)
print("Reserved test season:", TEST_SEASON, "(not scored)")

Saved: artifacts/logistic-regression-2023-2025
Training seasons: [2023, 2024]
Validation season: 2025
Reserved test season: 2026 (not scored)
